In [6]:
!pip install torch onnx -q

In [2]:
!pip install qai_hub_models qai-hub -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
!qai-hub configure --api_token uao5ieiewjzph11pbrr69canle5rdnbrob7kbxg2

2026-09-26 19:25:01.594 - INFO - Enabling verbose logging.
qai-hub configuration saved to /home/codespace/.qai_hub/client.ini
==================== /home/codespace/.qai_hub/client.ini ====================
[api]
api_token = uao5ieiewjzph11pbrr69canle5rdnbrob7kbxg2
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




In [3]:
"""
train_model.py

Trains a small autoencoder to learn a user's "normal" keystroke + sensor
pattern. At inference time, high reconstruction error = anomaly = possible
early stroke sign.

Feature vector (8 values per sample):
  [0] avg_dwell_time       - avg time key is held down (ms)
  [1] avg_flight_time      - avg time between key release and next key press (ms)
  [2] typing_speed         - keys per second
  [3] error_rate           - backspace/corrections per 100 keys
  [4] dwell_variance       - consistency of dwell time
  [5] flight_variance      - consistency of flight time
  [6] grip_strength        - from Arduino flex/grip sensor (normalized 0-1)
  [7] grip_asymmetry       - difference between left/right hand grip, if measured

Replace generate_synthetic_baseline() with real data pulled from your
Supabase `typing_baselines` / `sensor_baselines` tables once calibration
is actually saving data.
"""

import numpy as np
import torch
import torch.nn as nn

FEATURE_DIM = 8
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)


class KeystrokeAutoencoder(nn.Module):
    """Small autoencoder: learns to reconstruct 'normal' feature vectors.
    Anomalies (unfamiliar patterns) reconstruct poorly -> high error."""

    def __init__(self, input_dim=FEATURE_DIM, latent_dim=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 6),
            nn.ReLU(),
            nn.Linear(6, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 6),
            nn.ReLU(),
            nn.Linear(6, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out


def generate_synthetic_baseline(n_samples=500):
    """
    Generates fake 'normal' typing/sensor data to train on before you have
    real calibration data. Centered around plausible values with realistic
    noise. SWAP THIS OUT for real user data as soon as you have it.
    """
    dwell = np.random.normal(120, 15, n_samples)          # ms
    flight = np.random.normal(180, 25, n_samples)          # ms
    speed = np.random.normal(4.5, 0.6, n_samples)           # keys/sec
    error_rate = np.random.normal(2.0, 0.8, n_samples)      # per 100 keys
    dwell_var = np.random.normal(10, 3, n_samples)
    flight_var = np.random.normal(15, 4, n_samples)
    grip = np.random.normal(0.75, 0.08, n_samples)          # normalized
    grip_asym = np.random.normal(0.05, 0.02, n_samples)     # small normally

    data = np.stack(
        [dwell, flight, speed, error_rate, dwell_var, flight_var, grip, grip_asym],
        axis=1,
    ).astype(np.float32)
    return data


def normalize(data, mean=None, std=None):
    if mean is None:
        mean = data.mean(axis=0)
    if std is None:
        std = data.std(axis=0) + 1e-6
    return (data - mean) / std, mean, std


def train():
    # 1. Get data (synthetic for now)
    raw_data = generate_synthetic_baseline(n_samples=500)
    data, mean, std = normalize(raw_data)
    tensor_data = torch.tensor(data, dtype=torch.float32)

    # 2. Split train/val
    n_val = 50
    train_data = tensor_data[:-n_val]
    val_data = tensor_data[-n_val:]

    # 3. Model + optimizer
    model = KeystrokeAutoencoder()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    # 4. Train
    epochs = 200
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(train_data)
        loss = loss_fn(out, train_data)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            model.eval()
            with torch.no_grad():
                val_out = model(val_data)
                val_loss = loss_fn(val_out, val_data)
            print(f"Epoch {epoch:3d} | train_loss={loss.item():.4f} | val_loss={val_loss.item():.4f}")

    # 5. Compute anomaly threshold from validation reconstruction error
    model.eval()
    with torch.no_grad():
        val_out = model(val_data)
        per_sample_error = ((val_out - val_data) ** 2).mean(dim=1)
        threshold = per_sample_error.mean().item() + 3 * per_sample_error.std().item()

    print(f"\nSuggested anomaly threshold (reconstruction MSE): {threshold:.4f}")

    # 6. Save model + normalization stats + threshold together
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "feature_mean": mean,
            "feature_std": std,
            "anomaly_threshold": threshold,
        },
        "keystroke_autoencoder.pt",
    )
    print("Saved: keystroke_autoencoder.pt")

    return model, mean, std, threshold


if __name__ == "__main__":
    train()

Epoch   0 | train_loss=1.0644 | val_loss=1.1964
Epoch  20 | train_loss=0.9420 | val_loss=1.0684
Epoch  40 | train_loss=0.8788 | val_loss=1.0158
Epoch  60 | train_loss=0.7843 | val_loss=0.9428
Epoch  80 | train_loss=0.7398 | val_loss=0.9018
Epoch 100 | train_loss=0.7093 | val_loss=0.8775
Epoch 120 | train_loss=0.6895 | val_loss=0.8658
Epoch 140 | train_loss=0.6770 | val_loss=0.8747
Epoch 160 | train_loss=0.6706 | val_loss=0.8803
Epoch 180 | train_loss=0.6665 | val_loss=0.8817

Suggested anomaly threshold (reconstruction MSE): 2.4398
Saved: keystroke_autoencoder.pt


In [4]:
"""
export_onnx.py

Loads the trained autoencoder (keystroke_autoencoder.pt) and exports it to
ONNX format, ready to be optimized via Qualcomm AI Hub and run on-device
with ONNX Runtime + the QNN execution provider.

Run this AFTER train_model.py has produced keystroke_autoencoder.pt.
"""

import json

import torch

from train_model import KeystrokeAutoencoder, FEATURE_DIM

CHECKPOINT_PATH = "keystroke_autoencoder.pt"
ONNX_OUTPUT_PATH = "keystroke_autoencoder.onnx"
STATS_OUTPUT_PATH = "model_stats.json"


def export():
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)

    model = KeystrokeAutoencoder()
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Dummy input matching (batch_size=1, FEATURE_DIM) shape
    dummy_input = torch.randn(1, FEATURE_DIM, dtype=torch.float32)

    torch.onnx.export(
        model,
        dummy_input,
        ONNX_OUTPUT_PATH,
        input_names=["features"],
        output_names=["reconstruction"],
        dynamic_axes={"features": {0: "batch_size"}, "reconstruction": {0: "batch_size"}},
        opset_version=17,
    )
    print(f"Exported ONNX model to: {ONNX_OUTPUT_PATH}")

    # Save normalization stats + threshold separately as JSON so the native
    # client can use them without needing PyTorch installed at inference time
    stats = {
        "feature_mean": checkpoint["feature_mean"].tolist(),
        "feature_std": checkpoint["feature_std"].tolist(),
        "anomaly_threshold": checkpoint["anomaly_threshold"],
        "feature_order": [
            "avg_dwell_time",
            "avg_flight_time",
            "typing_speed",
            "error_rate",
            "dwell_variance",
            "flight_variance",
            "grip_strength",
            "grip_asymmetry",
        ],
    }
    with open(STATS_OUTPUT_PATH, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"Saved normalization stats + threshold to: {STATS_OUTPUT_PATH}")

    print("\nNext step: optimize this .onnx file via Qualcomm AI Hub")
    print("  pip install qai_hub_models qai-hub")
    print("  qai-hub configure --api_token <YOUR_API_TOKEN>")
    print("  (then submit a compile/profile job targeting your Snapdragon device)")


if __name__ == "__main__":
    export()

/tmp/ipykernel_12044/947028138.py:32: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0926 19:09:02.866000 12044 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0926 19:09:03.985000 12044 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0926 19:09:03.986000 12044 site-packages/torch/onnx/_inter

[torch.onnx] Obtain model graph for `KeystrokeAutoencoder([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `KeystrokeAutoencoder([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX model to: keystroke_autoencoder.onnx
Saved normalization stats + threshold to: model_stats.json

Next step: optimize this .onnx file via Qualcomm AI Hub
  pip install qai_hub_models qai-hub
  qai-hub configure --api_token <YOUR_API_TOKEN>
  (then submit a compile/profile job targeting your Snapdragon device)


In [1]:
import sys
print(sys.version)

3.11.16 (main, Aug 13 2026, 09:46:35) [GCC 13.3.0]


In [9]:
import qai_hub as hub

device = hub.Device("Snapdragon X Plus 8-Core CRD")

compile_job = hub.submit_compile_job(
    model="keystroke_autoencoder.onnx",
    device=device,
    options="--target_runtime onnx",
    input_specs={"features": (1, 8)}  # batch size 1, 8 features per your model
)

print(compile_job)

Uploading keystroke_autoencoder.onnx: 100%|███████████| 2.15k/2.15k [00:00<00:00, 2.73kB/s]


Scheduled compile job (jp3zrlnn5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp3zrlnn5/

Job(job_id=jp3zrlnn5, model_id=mnz0pj0xm, device=Device(name='Snapdragon X Plus 8-Core CRD', os='11', attributes=['os:windows', 'format:compute', 'framework:onnx', 'vendor:qualcomm', 'abi:aarch64-windows', 'chipset:qualcomm-snapdragon-x-plus-8-core', 'chipset:sc8340xp', 'hexagon:v73', 'soc-model:60', 'htp-supports-fp16:true', 'htp-supports-weight-sharing:true', 'framework:qnn']))


In [10]:
compile_job.wait()  # blocks until the job finishes
target_model = compile_job.get_target_model()
target_model.download("keystroke_autoencoder_optimized.onnx")
print("Downloaded optimized model")

keystroke_autoencoder_optimized.onnx.onnx.zip: 100%|██| 1.46k/1.46k [00:01<00:00, 1.02kB/s]

Downloaded model to keystroke_autoencoder_optimized.onnx.onnx.zip
Downloaded optimized model


In [11]:
import zipfile

with zipfile.ZipFile("keystroke_autoencoder_optimized.onnx.onnx.zip", "r") as z:
    z.extractall("optimized_model")

import os
print(os.listdir("optimized_model"))

['job_jp3zrlnn5_optimized_onnx']
